In [1]:
import pyrosetta
pyrosetta.init()
from pyrosetta.rosetta.protocols.relax import FastRelax
from pyrosetta.toolbox.mutants import mutate_residue
from pyrosetta import pose_from_pdb
from pyrosetta.rosetta.protocols.minimization_packing import PackRotamersMover, MinMover
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta import create_score_function
from pyrosetta import init, pose_from_pdb
from pyrosetta.rosetta.core.pose import PDBInfo
import pandas as pd
import plotly.graph_objects as go
import tqdm

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.Release.python313.ubuntu 2025.25+release.a0cefad01b3959ae8327a8931f5ad8c3fad27ea9 2025-06-18T10:51:52] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python313.ubuntu r404 2025.25+release.a0cefad01b a0

In [2]:
def mutate_pose(pose, position, new_residue):
    """
    Mutate a pose at a specific position to a new residue.
    """
    mutant_pose = pose.clone()
    mutate_residue(mutant_pose, position, new_residue)
    return mutant_pose  

def prepare_pose(pose, scorefxn):
    # pack sidechains around mutation site
    task = pyrosetta.standard_packer_task(pose)
    task.restrict_to_repacking()
    task.or_include_current(True)
    packer = PackRotamersMover(scorefxn, task)
    packer.apply(pose)
    
    # relax the pose
    relax = FastRelax()
    relax.set_scorefxn(scorefxn)
    relax.apply(pose)

    # minimize the pose
    movemap = MoveMap()
    movemap.set_bb(True)  # Allow backbone movements
    movemap.set_chi(True)  # Allow sidechain movements
    
    min_mover = MinMover()
    min_mover.movemap(movemap)
    min_mover.score_function(scorefxn)
    min_mover.apply(pose)

    return pose

def renumber_pdb_residues(input_pdb, output_pdb, start=1):
    init(extra_options="-ignore_unrecognized_res true")  # Initialize PyRosetta

    pose = pose_from_pdb(input_pdb)

    # Create a new PDBInfo object and assign renumbered info
    pdb_info = PDBInfo(pose)
    for i in range(1, pose.total_residue() + 1):
        pdb_info.set_resinfo(i, pose.pdb_info().chain(i), start + i - 1)

    pose.pdb_info(pdb_info)  # Replace with modified PDBInfo
    pose.dump_pdb(output_pdb)


def create_mutantslist(dataset_file, out_folder):
    mutants = []
    with open(dataset_file, "r") as f:
        lines = f.readlines()

        for line in lines[1:]:
            mutant = line.split(",")[0]
            mutants.append(mutant.strip())

    with open(f"{out_folder}/individual_list.txt", "w") as out:
        for mutant in mutants:
            mutant = mutant.split(":")
            chained_mutations = []
            for mutation in mutant:
                chained_mut = mutation[:1] + "A" + mutation[1:]
                chained_mutations.append(chained_mut)
            mutant = ",".join(chained_mutations)
            out.write(mutant +";" + "\n")

def retrieve_ddG(diff_fxout_file):
    ddG = []
    with open(diff_fxout_file, "r") as f:
        lines = f.readlines()
        for line in lines[9:]:
            line = line.split("\t")
            ddG.append((line[0], round(float(line[1]),4)))
            
    return ddG


In [14]:
data_set = "YAP1"
infile = f"/home/iwe80/Documents/Enzyme_Activity_Prediction/02_MAP/Data/Protein_Gym_Datasets/{data_set}.csv"
out_folder = f"/home/iwe80/Documents/Enzyme_Activity_Prediction/02_MAP/Data/Protein_Gym_Datasets/DeltaDeltaG/{data_set}"
create_mutantslist(infile, out_folder)

In [ ]:
"""Update PDB file with DeltaDeltaG values"""
diff_fxout_file = "/home/iwe80/Documents/Enzyme_Activity_Prediction/02_Playground/MAP/DeltaDeltaG/GRB2/Dif_AF_P62993_F1_model_v4.fxout"
ddG = retrieve_ddG(diff_fxout_file)

mutants_file = "/home/iwe80/Documents/Enzyme_Activity_Prediction/02_Playground/MAP/Data/Protein_Gym_Datasets/D7PM05.csv"

mutants_df = pd.read_csv(mutants_file)
# Insert ΔΔG column at the end (or at position 6 if you know the structure)
mutants_df.insert(6 if mutants_df.shape[1] >= 6 else mutants_df.shape[1], "ΔΔG", [val[1] for val in ddG])

# Save the modified DataFrame back to a CSV file
mutants_df.to_csv(mutants_file, index=False)

ValueError: cannot insert ΔΔG, already exists

In [9]:
fig = go.Figure(
    data=go.Scatter(
        x=mutants_df["Norm_score_1"],
        y=mutants_df["ΔΔG"],
        mode='markers',
        marker=dict(size=3, opacity=0.5)
    )
)
fig.update_layout(
    yaxis_title="ΔΔG",
    xaxis_title="Norm_score_1",
    title=f"Norm_score_1 vs ΔΔG for {mutants_file.split('/')[-1].split(".")[0]}-Dataset")
fig.show()

In [19]:
import numpy as np

# Bin the Norm_score_1 values of the filtered dataframe
threshold = 0.15
score_bins = 40
bin_size = round((mutants_df["Norm_score_1"].max() - mutants_df["Norm_score_1"].min()) / score_bins,3)
# Define bin edges from 0 to 1 in steps of 0.05
bin_edges = np.arange(0, (1 + bin_size), bin_size)
# bin_labels = [f"({round(bin_edges[i], 3)}, {round(bin_edges[i+1], 3)}]" for i in range(len(bin_edges)-1)]

filtered_df = mutants_df.nsmallest(int(threshold * len(mutants_df)), "ΔΔG").copy()
filtered_df['score_bin'] = pd.cut(filtered_df['Norm_score_1'], bins=bin_edges, include_lowest=False) #labels=bin_labels, 

# Count the number of points in each score bin
score_bin_counts = filtered_df['score_bin'].value_counts().sort_index()

# Plot as a bar chart
fig = go.Figure(
    data=go.Bar(
        x=score_bin_counts.index.astype(str),
        y=score_bin_counts.values,
        text=score_bin_counts.values,
        textposition='outside'
    )
)

fig.update_layout(
    xaxis_title=f"Norm_score_1 bins (lowest {int(threshold*100)}% ΔΔG, bin size 0.05)",
    yaxis_title="Count",
    title=f"Distribution of Norm_score_1 (lowest {int(threshold*100)}% ΔΔG,, bin size 0.05)"
)
fig.show()

In [ ]:
# mutants_file = "/home/iwe80/Documents/Enzyme_Activity_Prediction/02_Playground/MAP/Data/Protein_Gym_Datasets/D7PM05.csv"
# out_folder = "/home/iwe80/Documents/DeltaDeltaG/D7PM05/"


In [25]:
"""mutate Sequence"""
wt_seq = "MDPGQQPPPQPAPQGQGQPPSQPPQGQGPPSGPGQPAPAATQAAPQAPPAGHQIVHVRGDSETDLEALFNAVMNPKTANVPQTVPMRLRKLPDSFFKPPEPKSHSRQASTDAGTAGALTPQHVRAHSSPASLQLGAVSPGTLTPTGVVSGPAATPTAQHLRQSSFEIPDDVPLPAGWEMAKTSSGQRYFLNHIDQTTTWQDPRKAMLSQMNVTAPTSPPVQQNMMNSASGPLPDGWEQAMTQDGEIYYINHKNKTTSWLDPRLDPRFAMNQRISQSAPVKQPPPLAPQSPQGGVMGGSNSNQQQQMRLQQLQMEKERLRLKQQELLRQAMRNINPSTANSPKCQELALRSQLPTLEQDGGTQNPVSSPGMSQELRTMTTNSSDPFLNSGTYHSRDESTDSGLSMSSYSVPRTPDDFLNSVDEMDTGDTINQSTLPSQQNRFPDYLEAIPGTNVDLGTLEGDGMNIEGEELMPSLQEALSSDILNDMESVLAATKLDKESFLTWL"
in_file = "/home/iwe80/Documents/Enzyme_Activity_Prediction/02_MAP/Data/Protein_Gym_Datasets/YAP1.csv"
out_file = f"{in_file[:-4]}_corrected.csv"

with open(in_file, "r") as f:
    lines = f.readlines()

with open(out_file, "w") as out:
    out.write(lines[0])  # Write header
    for line in lines[1:]:
        line = line.split(",")[:-1]
        mutant = line[0]
        line[1] = wt_seq
        try:
            for mutation in mutant.split(":"):
                position = int(mutation[1:-1])-1
                new_residue = mutation[-1]
                line[1] = line[1][:position] + new_residue + line[1][position + 1:]
        except Exception as e:
            pass
        out.write(",".join(line)+"\n")

    

